## Parameter Management

In [26]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(), nn.LazyLinear(1))
X = torch.rand(size=(2, 4))
net(X).shape, " ",net(X) 

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


(torch.Size([2, 1]),
 ' ',
 tensor([[0.3587],
         [0.3255]], grad_fn=<AddmmBackward0>))

### Parameter Access

In [3]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[-0.0186, -0.0536,  0.2708, -0.1290,  0.1575, -0.2258,  0.2334, -0.1964]])),
             ('bias', tensor([0.0692]))])

In [6]:
net[0].state_dict(), net[1].state_dict()

(OrderedDict([('weight',
               tensor([[ 0.2054, -0.0699,  0.2789, -0.2106],
                       [-0.3308, -0.2348, -0.3721,  0.2529],
                       [-0.3883,  0.4251,  0.2190, -0.0969],
                       [ 0.1251,  0.2958, -0.3851,  0.0552],
                       [ 0.3255,  0.1905,  0.4715,  0.2193],
                       [-0.3688, -0.0132, -0.0827, -0.2009],
                       [ 0.2311, -0.2324,  0.2342, -0.1035],
                       [-0.4296, -0.3530, -0.4621, -0.0830]])),
              ('bias',
               tensor([ 0.0829, -0.0824,  0.4587,  0.0812, -0.1685, -0.4850,  0.3540,  0.2416]))]),
 OrderedDict())

### Targeted Parameters

In [10]:
type(net[2].bias), net[2].bias.data

(torch.nn.parameter.Parameter, tensor([0.0692]))

In [12]:
type(net[2].weight), net[2].weight.data

(torch.nn.parameter.Parameter,
 tensor([[-0.0186, -0.0536,  0.2708, -0.1290,  0.1575, -0.2258,  0.2334, -0.1964]]))

In [13]:
net[2].weight.grad == None

True

### All parameters at Once

In [16]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([1, 8])),
 ('2.bias', torch.Size([1]))]

In [18]:
list(net.named_parameters())

[('0.weight',
  Parameter containing:
  tensor([[ 0.2054, -0.0699,  0.2789, -0.2106],
          [-0.3308, -0.2348, -0.3721,  0.2529],
          [-0.3883,  0.4251,  0.2190, -0.0969],
          [ 0.1251,  0.2958, -0.3851,  0.0552],
          [ 0.3255,  0.1905,  0.4715,  0.2193],
          [-0.3688, -0.0132, -0.0827, -0.2009],
          [ 0.2311, -0.2324,  0.2342, -0.1035],
          [-0.4296, -0.3530, -0.4621, -0.0830]], requires_grad=True)),
 ('0.bias',
  Parameter containing:
  tensor([ 0.0829, -0.0824,  0.4587,  0.0812, -0.1685, -0.4850,  0.3540,  0.2416],
         requires_grad=True)),
 ('2.weight',
  Parameter containing:
  tensor([[-0.0186, -0.0536,  0.2708, -0.1290,  0.1575, -0.2258,  0.2334, -0.1964]],
         requires_grad=True)),
 ('2.bias',
  Parameter containing:
  tensor([0.0692], requires_grad=True))]

### Tied Parameters

In [19]:
# We need to give the shared layer a name so that we can refer to its
# parameters
shared = nn.LazyLinear(8)
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.LazyLinear(1))
net(X)

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


tensor([[-0.4113],
        [-0.4039]], grad_fn=<AddmmBackward0>)

In [20]:
# Check whether the parameters are the same
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100

tensor([True, True, True, True, True, True, True, True])


In [21]:
# Make sure that they are actually the same object rather than just having the
# same value
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])


## Exercises

In [22]:
X = torch.rand(2, 20)

In [24]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Random weight parameters that will not compute gradients and
        # therefore keep constant during training
        self.rand_weight = torch.rand((20, 20))
        self.linear = nn.LazyLinear(20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(X @ self.rand_weight + 1)
        # Reuse the fully connected layer. This is equivalent to sharing
        # parameters with two fully connected layers
        X = self.linear(X)
        # Control flow
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [27]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.LazyLinear(64), nn.ReLU(),
                                 nn.LazyLinear(32), nn.ReLU())
        self.linear = nn.LazyLinear(16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.LazyLinear(20), FixedHiddenMLP())
chimera(X)

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


tensor(-0.4778, grad_fn=<SumBackward0>)

In [28]:
list(chimera.named_parameters())

[('0.net.0.weight',
  Parameter containing:
  tensor([[-0.2076,  0.0918,  0.2038,  ..., -0.0431,  0.0466,  0.1957],
          [ 0.0306,  0.0081,  0.1532,  ..., -0.1193, -0.0313, -0.0663],
          [ 0.1435,  0.0816,  0.1284,  ...,  0.1222,  0.1420, -0.1645],
          ...,
          [ 0.0049, -0.2129,  0.0478,  ...,  0.1740, -0.0364, -0.0291],
          [-0.1489,  0.0594, -0.1648,  ..., -0.1240, -0.2088,  0.2193],
          [ 0.1161, -0.1254, -0.0314,  ...,  0.1746, -0.2209, -0.0308]],
         requires_grad=True)),
 ('0.net.0.bias',
  Parameter containing:
  tensor([-0.0280, -0.1516, -0.1043,  0.1716, -0.1850,  0.0695,  0.1091, -0.0874,
          -0.0841,  0.0150, -0.0697, -0.0996,  0.0762, -0.1238, -0.1880, -0.1142,
          -0.1764, -0.0585, -0.0075,  0.0740,  0.0270, -0.0926,  0.1913, -0.2138,
          -0.1555,  0.1083,  0.1870, -0.0193,  0.0329, -0.0230,  0.0425,  0.1890,
          -0.2112,  0.0867,  0.1909,  0.0233, -0.0153, -0.1043,  0.1104, -0.1213,
          -0.1267,  0.104

In [29]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([8, 8])),
 ('2.bias', torch.Size([8])),
 ('6.weight', torch.Size([1, 8])),
 ('6.bias', torch.Size([1]))]